In [14]:
# Import packages.
import pandas as pd
import numpy as np
import os



In [ ]:
# Load data.

# define file path, separate for organization.
# path for Mac
#netflix_prize_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/combined_data_1.txt"

# Path for Pc
netflix_prize_filepath = "D:/data/Netflix/combined_data_1.txt"


# data is separated by "," so we can use pd.read_csv().
rows = []
movie_id = None

with open(netflix_prize_filepath, "r") as file:
    for line in file:
        line = line.strip()

        if line.endswith(":"):
            movie_id = line.replace(":", "")
        
        else:
            customer_id, movie_rating, date = line.split(",")
            rows.append([movie_id, customer_id, movie_rating, date ])
            
netflix_prize_data = pd.DataFrame(
    rows,
    columns = ["movie_id", "customer_id", "movie_rating", "date"]  
)

# Convert data to useable format.
netflix_prize_data["movie_id"] = netflix_prize_data["movie_id"].astype(int)
netflix_prize_data["customer_id"] = netflix_prize_data["customer_id"].astype(int)
netflix_prize_data["customer_movie_rating"] = netflix_prize_data["movie_rating"].astype(float)
netflix_prize_data["rating_date"] = pd.to_datetime(netflix_prize_data["date"])
netflix_prize_data = netflix_prize_data.drop(columns=["date","movie_rating"])

# Preview first couple rows.
netflix_prize_data.head()

In [ ]:
# Feature Engineer.
netflix_prize_data["avg_customer_rating"] = netflix_prize_data.groupby("customer_id")["customer_movie_rating"].transform("mean")
netflix_prize_data["avg_movie_rating"] = netflix_prize_data.groupby("movie_id")["customer_movie_rating"].transform("mean")
netflix_prize_data["avg_date_rating"] = netflix_prize_data.groupby("rating_date")["customer_movie_rating"].transform("mean")
netflix_prize_data["count_of_ratings_per_date"] = netflix_prize_data.groupby("rating_date")["customer_movie_rating"].transform("count")
netflix_prize_data["count_of_ratings_per_movie"] = netflix_prize_data.groupby("movie_id")["customer_movie_rating"].transform("count")

# Preview.
netflix_prize_data.head()

In [ ]:
# Load Netflix movie titles.

# define file path, separate for organization.
# Path for Mac
#netflix_titles_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/movie_titles.csv"

#Path for Pc
netflix_titles_filepath = "D:/data/Netflix/movie_titles.csv"

netflix_titles_data = pd.read_csv(
    netflix_titles_filepath,
    encoding="latin-1",
    header=None,
    names=["movie_id", "year", "title"],
    sep=",",
    engine="python",
    on_bad_lines="skip")

# Convert data to useable format.
netflix_titles_data["movie_id"] = netflix_titles_data["movie_id"].astype(int)
netflix_titles_data["release_year"] = pd.to_numeric(
    netflix_titles_data["year"],
    errors="coerce"
)
netflix_titles_data = netflix_titles_data.drop(columns="year")

netflix_titles_data["title"] = netflix_titles_data["title"].str.strip()

# checking duplicate titles returned "False", tells us we need to look further into the matter as movies can have duplicate titles with different release years.
netflix_titles_data["title"].is_unique
netflix_titles_data["movie_id"].is_unique

# Further inspection confirms that duplicate movies are released in separate years.
netflix_titles_data[
    netflix_titles_data["title"].duplicated(keep=False)
].sort_values("title")


# Preview first couple rows.
netflix_titles_data.head()

In [ ]:
netflix_merged_data = netflix_prize_data.merge(
    netflix_titles_data,
    on= "movie_id",
    how= "left"
)

netflix_merged_data.head()


In [ ]:
# Feature engineer
netflix_merged_data["first_rating_date"] = (netflix_merged_data.groupby("customer_id")["rating_date"].transform("min"))
netflix_merged_data["last_rating_date"] = (netflix_merged_data.groupby("customer_id")["rating_date"].transform("max"))
netflix_merged_data["customer_active_days"] = (
    netflix_merged_data["last_rating_date"] - netflix_merged_data["first_rating_date"]
).dt.days

netflix_merged_data = netflix_merged_data.sort_values(["customer_id", "rating_date"])
netflix_merged_data["days_since_previous_rating"] = (
    netflix_merged_data.groupby("customer_id")["rating_date"].diff().dt.days
)
netflix_merged_data["count_of_ratings_per_customer"] = netflix_merged_data.groupby("customer_id")["customer_movie_rating"].transform("count")


netflix_merged_data["day_of_week"] = (
    netflix_merged_data["rating_date"].dt.day_name()
)

netflix_merged_data["is_weekend"] = (
    netflix_merged_data["rating_date"].dt.weekday >= 5
).astype(int)


netflix_merged_data["release_decade"] = (
    (netflix_merged_data["release_year"] // 10) * 10
)

netflix_merged_data["movie_age_at_rating"] = (
    (netflix_merged_data["rating_date"].dt.year) - netflix_merged_data["release_year"]
)

netflix_merged_data["avg_movie_rating_count_for_customer"] = (
    netflix_merged_data.groupby("customer_id")["count_of_ratings_per_movie"].transform("mean")
)

netflix_merged_data["customer_movie_rating_count_diff"] = (
    netflix_merged_data["count_of_ratings_per_movie"] - netflix_merged_data["avg_movie_rating_count_for_customer"]
)


netflix_merged_data.head()